# Potter Airlines — SQLite Database

This notebook connects the generated `flights.json` dataset to a SQLite database and provides parameterized INSERT, SELECT, UPDATE, DELETE operations.


In [16]:
import sqlite3
import json

DB_NAME = "potter_airlines.db"
JSON_FILE = "flights.json"


## 1. Create the Flights Table


In [4]:
def create_table():
    """Create the flights table if it does not already exist."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS flights (
                flight_id TEXT PRIMARY KEY,
                origin TEXT NOT NULL,
                destination TEXT NOT NULL,
                departure_date TEXT NOT NULL,
                days_until_departure INTEGER NOT NULL
                    CHECK (days_until_departure >= 0),
                base_fare REAL NOT NULL
                    CHECK (base_fare > 0),
                seats_remaining INTEGER NOT NULL
                    CHECK (seats_remaining >= 0 AND seats_remaining <= capacity),
                capacity INTEGER NOT NULL
                    CHECK (capacity > 0),
                route_popularity REAL NOT NULL
                    CHECK (route_popularity >= 0 AND route_popularity <= 1),
                international INTEGER NOT NULL
                    CHECK (international IN (0, 1))
            )
        """)


## 2. Load Flight Data from JSON


In [5]:
def load_json(filename=JSON_FILE):
    """Load flight data from the JSON file."""

    with open(filename, "r") as file:
        return json.load(file)


## 3. INSERT — One Flight


In [8]:
def insert_flight(flight):
    """Insert one flight into the database."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            INSERT INTO flights
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            flight["flight_id"],
            flight["origin"],
            flight["destination"],
            flight["departure_date"],
            flight["days_until_departure"],
            flight["base_fare"],
            flight["seats_remaining"],
            flight["capacity"],
            flight["route_popularity"],
            flight["international"]
        ))


## 4. Insert All Flights from JSON


In [9]:
def insert_all_flights(flights):
    """Insert all flights from the JSON dataset."""

    with sqlite3.connect(DB_NAME) as conn:
        for flight in flights:
            conn.execute("""
                INSERT OR IGNORE INTO flights
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                flight["flight_id"],
                flight["origin"],
                flight["destination"],
                flight["departure_date"],
                flight["days_until_departure"],
                flight["base_fare"],
                flight["seats_remaining"],
                flight["capacity"],
                flight["route_popularity"],
                flight["international"]
            ))


## 5. SELECT — One Flight


In [10]:
def get_flight(flight_id):
    """Retrieve one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            SELECT * FROM flights
            WHERE flight_id = ?
        """, (flight_id,))
        return cursor.fetchone()


## 6. SELECT — All Flights


In [11]:
def get_all_flights():
    """Retrieve all flights from the database."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("SELECT * FROM flights")
        return cursor.fetchall()


## 7. UPDATE — Seats Remaining

This update first checks that the flight exists and that the new seat count is between 0 and the flight capacity.


In [13]:
def update_seats(flight_id, new_seats):
    """Update seats remaining after validating the flight and seat count."""

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.execute("""
            SELECT capacity
            FROM flights
            WHERE flight_id = ?
        """, (flight_id,))

        result = cursor.fetchone()

        if result is None:
            raise ValueError("Flight not found.")

        capacity = result[0]

        if new_seats < 0 or new_seats > capacity:
            raise ValueError("Seats remaining must be between 0 and capacity.")

        conn.execute("""
            UPDATE flights
            SET seats_remaining = ?
            WHERE flight_id = ?
        """, (new_seats, flight_id))


## 8. DELETE — One Flight


In [14]:
def delete_flight(flight_id):
    """Delete one flight by flight ID."""

    with sqlite3.connect(DB_NAME) as conn:
        conn.execute("""
            DELETE FROM flights
            WHERE flight_id = ?
        """, (flight_id,))


## 9. Set Up Database and Validate


In [15]:
create_table()

flights = load_json()
insert_all_flights(flights)

print(f"{len(flights)} flights loaded from JSON.")
print(f"{len(get_all_flights())} flights stored in SQLite.")

# Check that all flights were successfully stored
assert len(get_all_flights()) == len(flights)

print("Database setup completed successfully.")


1044 flights loaded from JSON.
1044 flights stored in SQLite.
Database setup completed successfully.


## Summary

The flight data from `flights.json` is stored in a SQLite database called `potter_airlines.db`. The `flights` table follows the same structure as the JSON dataset so that the data can be transferred directly into the database. Parameterized SQL queries are used to insert and retrieve flight records, update remaining seats, and delete flights when needed. Basic database constraints and update validation are included to prevent invalid values such as negative capacity, invalid seat counts, route popularity outside 0–1, or an invalid international indicator. This allows the project to store flight information persistently and access or modify specific records efficiently.
